# 02 · Clean

This notebook loads each source from `data/raw/` and fixes what is wrong with it. Save one clean table for each source in `data/interim/`,
and record each problem and its fix in `docs/cleaning_log.md`.

In [1]:
import json
from io import StringIO
from pathlib import Path

import pandas as pd

RAW = Path("../data/raw")
INTERIM = Path("../data/interim")
INTERIM.mkdir(parents=True, exist_ok=True)

## Source 1

In [2]:
oecd = pd.read_csv(RAW / "oecd_share_prices.csv")
print(oecd.shape)
oecd.head()

(528, 16)


,DATAFLOW,REF_AREA,FREQ,MEASURE,UNIT_MEASURE,ACTIVITY,ADJUSTMENT,TRANSFORMATION,TIME_HORIZ,METHODOLOGY,TIME_PERIOD,OBS_VALUE,OBS_STATUS,UNIT_MULT,DECIMALS,BASE_PER
0,OECD.SDD.STES:DSD_STES@DF_FINMARK(4.0),AUS,A,SHARE,IX,_Z,_Z,_Z,_Z,N,2014,98.61095,A,0,2,2015
1,OECD.SDD.STES:DSD_STES@DF_FINMARK(4.0),AUS,A,SHARE,IX,_Z,_Z,_Z,_Z,N,2015,100.00000,A,0,2,2015
2,OECD.SDD.STES:DSD_STES@DF_FINMARK(4.0),AUS,A,SHARE,IX,_Z,_Z,_Z,_Z,N,2016,96.06994,A,0,2,2015
3,OECD.SDD.STES:DSD_STES@DF_FINMARK(4.0),AUS,A,SHARE,IX,_Z,_Z,_Z,_Z,N,2017,105.64780,A,0,2,2015
4,OECD.SDD.STES:DSD_STES@DF_FINMARK(4.0),AUS,A,SHARE,IX,_Z,_Z,_Z,_Z,N,2018,109.20510,A,0,2,2015


In [3]:
oecd.dtypes

DATAFLOW              str
REF_AREA              str
FREQ                  str
MEASURE               str
UNIT_MEASURE          str
ACTIVITY              str
ADJUSTMENT            str
TRANSFORMATION        str
TIME_HORIZ            str
METHODOLOGY           str
TIME_PERIOD         int64
OBS_VALUE         float64
OBS_STATUS            str
UNIT_MULT           int64
DECIMALS            int64
BASE_PER            int64
dtype: object

In [4]:
for col in ["FREQ", "MEASURE", "UNIT_MEASURE", "ADJUSTMENT", "TRANSFORMATION", "METHODOLOGY"]:
    print(col, oecd[col].unique())

FREQ <ArrowStringArray>
['A']
Length: 1, dtype: str
MEASURE <ArrowStringArray>
['SHARE']
Length: 1, dtype: str
UNIT_MEASURE <ArrowStringArray>
['IX']
Length: 1, dtype: str
ADJUSTMENT <ArrowStringArray>
['_Z']
Length: 1, dtype: str
TRANSFORMATION <ArrowStringArray>
['_Z']
Length: 1, dtype: str
METHODOLOGY <ArrowStringArray>
['N']
Length: 1, dtype: str


In [5]:
oecd = oecd[oecd["UNIT_MEASURE"] == "IX"]

shares = pd.DataFrame({
    "country_code": oecd["REF_AREA"],
    "year": oecd["TIME_PERIOD"],
    "share_price_index": oecd["OBS_VALUE"],
})
shares = shares.sort_values(["country_code", "year"]).reset_index(drop=True)
shares.dtypes

country_code             str
year                   int64
share_price_index    float64
dtype: object

In [6]:
print(shares["country_code"].nunique(), "areas")
print(shares[shares.duplicated(["country_code", "year"], keep=False)])
shares["share_price_index"].describe()

48 areas
Empty DataFrame
Columns: [country_code, year, share_price_index]
Index: []


count     528.000000
mean      128.323659
std        67.163926
min        59.948520
25%       100.000000
50%       112.609000
75%       139.159575
max      1190.710000
Name: share_price_index, dtype: float64

In [7]:
shares.to_parquet(INTERIM / "share_prices.parquet", index=False)
shares.head()

,country_code,year,share_price_index
0,AUS,2014,98.61095
1,AUS,2015,100.00000
2,AUS,2016,96.06994
3,AUS,2017,105.64780
4,AUS,2018,109.20510


## Source 2

In [8]:
records = []
for path in sorted(RAW.glob("worldbank_inflation_page_*.json")):
    with open(path) as f:
        page = json.load(f)
    records += page[1]

print(len(records), "records, the API said", page[0]["total"])
records[0]

2926 records, the API said 2926


{'indicator': {'id': 'FP.CPI.TOTL.ZG',
  'value': 'Inflation, consumer prices (annual %)'},
 'country': {'id': 'ZH', 'value': 'Africa Eastern and Southern'},
 'countryiso3code': 'AFE',
 'date': '2024',
 'value': 4.77085731915833,
 'unit': '',
 'obs_status': '',
 'decimal': 1}

In [9]:
wb = pd.json_normalize(records)
print(wb.dtypes)
wb.head()

countryiso3code        str
date                   str
value              float64
unit                   str
obs_status             str
decimal              int64
indicator.id           str
indicator.value        str
country.id             str
country.value          str
dtype: object


,countryiso3code,date,value,unit,obs_status,decimal,indicator.id,indicator.value,country.id,country.value
0,AFE,2024,4.770857,,,1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",ZH,Africa Eastern and Southern
1,AFE,2023,7.399186,,,1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",ZH,Africa Eastern and Southern
2,AFE,2022,10.883478,,,1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",ZH,Africa Eastern and Southern
3,AFE,2021,6.824727,,,1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",ZH,Africa Eastern and Southern
4,AFE,2020,5.191629,,,1,FP.CPI.TOTL.ZG,"Inflation, consumer prices (annual %)",ZH,Africa Eastern and Southern


In [10]:
inflation = pd.DataFrame({
    "country_code": wb["countryiso3code"],
    "country": wb["country.value"],
    "year": wb["date"].astype(int),
    "inflation_pct": wb["value"],
})

print((inflation["country_code"] == "").sum(), "rows without a country code")
inflation = inflation[inflation["country_code"] != ""]

55 rows without a country code


In [11]:
names = {
    "Korea, Rep.": "South Korea",
    "Turkiye": "Türkiye",
    "Russian Federation": "Russia",
    "Slovak Republic": "Slovakia",
    "Czechia": "Czech Republic",
}
inflation["country"] = inflation["country"].replace(names)

inflation = inflation.sort_values(["country_code", "year"]).reset_index(drop=True)
print(inflation.duplicated(["country_code", "year"]).sum(), "duplicates")
print(inflation["inflation_pct"].isna().sum(), "missing values")
inflation["inflation_pct"].describe()

0 duplicates
376 missing values


count    2495.000000
mean        6.330540
std        21.102623
min        -6.687321
25%         1.414524
50%         3.070667
75%         5.816015
max       557.201817
Name: inflation_pct, dtype: float64

In [12]:
inflation.to_parquet(INTERIM / "inflation.parquet", index=False)
inflation.head()

,country_code,country,year,inflation_pct
0,ABW,Aruba,2014,0.421441
1,ABW,Aruba,2015,0.474764
2,ABW,Aruba,2016,-0.931196
3,ABW,Aruba,2017,-1.028282
4,ABW,Aruba,2018,3.626041


## Source 3

In [13]:
html = (RAW / "wikipedia_iso_3166_1.html").read_text(encoding="utf-8")

tables = pd.read_html(StringIO(html), match="Link to ISO 3166-2", keep_default_na=False)
print(len(tables), "table(s)")
wiki = tables[0]
print(wiki.shape)
wiki.head()

1 table(s)
(7, 6)


,Short name upper case in ISO 3166,Short name lower case in ISO 3166 Full name,Listed as independent in ISO 3166,Local short name,Dispute,Link to ISO 3166-2
0,AFGHANISTAN,Afghanistanthe Islamic Republic of Afghanistan,Yes[6],Afghānistān (ps) Afghānistān (fa),Under the unrecognized government of the Islam...,ISO 3166-2:AF
1,CHINA,China the People's Republic of China,Yes[7],Zhongguo (zh),Disputed sovereignty with Taiwan,ISO 3166-2:CN
2,CYPRUS,Cyprus the Republic of Cyprus,Yes[8],Kýpros (el) Kıbrıs (tr),Disputed sovereignty with Northern Cyprus,ISO 3166-2:CY
3,FALKLAND ISLANDS (MALVINAS),Falkland Islands (the) [Malvinas][a],No[9],Falkland Islands (the) [Malvinas] (en),Disputed sovereignty and naming,ISO 3166-2:FK
4,"PALESTINE, STATE OF","Palestine, State of the State of Palestine",No[10],Dawlat Filasţīn (ar),Disputed sovereignty with Israel,ISO 3166-2:PS


In [14]:
wiki.columns = ["name", "alpha2", "country_code", "numeric", "subdivisions", "independent"]

iso = wiki[["country_code", "alpha2"]].copy()
iso["country_code"] = iso["country_code"].str.strip()
iso["alpha2"] = iso["alpha2"].str.strip()

print(iso["country_code"].str.len().value_counts())
print(iso["country_code"].duplicated().sum(), "duplicate codes")
iso[iso["country_code"].isin(["NAM", "DEU", "GBR"])]

country_code
6    6
5    1
Name: count, dtype: int64
0 duplicate codes


,country_code,alpha2


In [15]:
iso.to_parquet(INTERIM / "country_codes.parquet", index=False)
len(iso)

7